# MCP & LLM Integration – Intelligent Factory Agents

**Internet of Production** · Prof. Dr.-Ing. U. Triltsch · Ostfalia University of Applied Sciences

---

## Learning objectives

After completing this notebook you are able to:

1. Explain the **MCP protocol** (Model Context Protocol) at the level of architecture and messages.
2. Implement a working **MCP server** that exposes factory telemetry as tools.
3. Write an **MCP client** that calls tools and processes the responses.
4. Integrate an **LLM as decision-making instance** into a LangGraph agent.
5. Demonstrate the difference between **rule-based** and **LLM-based** coordination logic on the same scenario.

---

## Classification within the lecture series

| Notebook | Paradigm | Decision logic | Connectivity |
|---|---|---|---|
| `W1` / `W2` Reactive / Communicative | Sense–decide–act | Rule-based | OPC UA (direct) |
| `01_LangGraph_Agents_solution.ipynb` | Stateful graph | Rule-based | Simulated state |
| **`02_MCP_LLM_solution.ipynb` (this notebook)** | **Stateful graph** | **LLM-based** | **MCP → OPC UA / REST** |

The factory scenario remains identical: machines heat up, a coordinator agent
decides about repairs, a maintenance agent carries them out. What is new:
- The coordinator uses an **LLM** instead of hard-coded rules.
- The telemetry data come via **MCP** instead of a simulated dict.

---

## Structure of this notebook

```
Part 1 – MCP protocol:        architecture, message format, tool schema
Part 2 – MCP server:          implementation with FastAPI + mcp library
Part 3 – MCP client:          tool discovery, tool call, response processing
Part 4 – LLM as agent:        stub implementation, real API call (extension)
Part 5 – Full integration:    LangGraph + MCP + LLM in the factory scenario
Part 6 – Comparison:          rule-based vs. LLM-based
```


---
## Installation


In [ ]:
# Core dependencies
!pip install -q "langgraph>=0.2" "langchain-core>=0.2"
!pip install -q "fastapi" "uvicorn" "httpx" "requests"

# MCP Python SDK (official Anthropic implementation)
!pip install -q "mcp>=1.0"

# For the real LLM connection (Part 4, extension)
# Alternative:
# for Claude
#!pip install -q "langchain-anthropic>=0.1"
# for GPT-4o
!pip install -q "langchain-openai>=0.1"

print("Installation finished.")

In [ ]:
import json
import random
import threading
import time
import textwrap
from typing import TypedDict, Dict, List, Literal, Any, Optional

import requests
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

from langgraph.graph import StateGraph, END

print("All imports successful.")

---
# Part 1 – MCP protocol: architecture and message format

## 1.1 What is MCP and why does it exist?

Before MCP every LLM application had to write its own integration code for each external
system: its own HTTP clients, its own authentication logic, its own
error handling. As the number of tools grew, an **N×M integration problem** emerged.

```
WITHOUT MCP (N×M integrations)       WITH MCP (N+M integrations)

Agent A ──► OPC UA (custom)          Agent A ──► MCP client ──► MCP server: OPC UA
Agent A ──► REST API (custom)                                ──► MCP server: REST
Agent A ──► DB (custom)                                      ──► MCP server: DB
Agent B ──► OPC UA (custom)          Agent B ──► MCP client ──► (same servers)
Agent B ──► REST API (custom)
```

MCP solves this problem by **standardisation at protocol level** — analogous to
how HTTP standardised the web.

## 1.2 Architecture: host, client, server

```
┌─────────────────────────────────────────────┐
│  HOST (LLM application / agent framework)   │
│                                             │
│  ┌─────────────┐   ┌─────────────────────┐  │
│  │  LLM        │◄──│  MCP client A       │  │
│  │ (reasoning) │   │  (OPC UA adapter)   │  │
│  └─────────────┘   └────────┬────────────┘  │
│                             │ JSON-RPC 2.0  │
└─────────────────────────────┼───────────────┘
                              │
              ┌───────────────▼──────────────┐
              │  MCP SERVER                  │
              │  (runs separately / remote)  │
              │                              │
              │  Tools:                      │
              │  - read_temperature(id)      │
              │  - write_temperature(id, v)  │
              │  - list_machines()           │
              └───────────────┬──────────────┘
                              │
                    ┌─────────▼──────────┐
                    │  OPC UA / REST /   │
                    │  database / PLC    │
                    └────────────────────┘
```

**Three roles in the MCP ecosystem:**

- **Host**: the LLM application (e.g. LangGraph graph, Claude Desktop). Contains the MCP client.
- **MCP client**: language library in the host that implements the MCP protocol. Manages connections to MCP servers.
- **MCP server**: exposes *tools*, *resources* or *prompts* via the MCP protocol. Encapsulates the access logic to the system behind it.

## 1.3 Message format: JSON-RPC 2.0

MCP is based on **JSON-RPC 2.0** — a lightweight remote procedure call protocol.
All messages are JSON objects with a defined structure.


In [ ]:
# ── Examples of MCP messages (JSON-RPC 2.0) ──────────────────────────────────
# These messages are normally generated by the MCP library.
# Inspecting them manually helps to understand the protocol.

# 1. Tool discovery: the client asks which tools the server offers
request_tools_list = {
    "jsonrpc": "2.0",
    "method": "tools/list",
    "params": {},
    "id": 1
}

# 2. Server response with tool definitions (schema)
response_tools_list = {
    "jsonrpc": "2.0",
    "result": {
        "tools": [
            {
                "name": "read_temperature",
                "description": "Reads the current temperature of a machine in °C.",
                "inputSchema": {
                    "type": "object",
                    "properties": {
                        "machine_id": {
                            "type": "string",
                            "description": "Machine identifier, e.g. 'M01'"
                        }
                    },
                    "required": ["machine_id"]
                }
            },
            {
                "name": "list_machines",
                "description": "Returns a list of all available machines.",
                "inputSchema": {
                    "type": "object",
                    "properties": {}
                }
            }
        ]
    },
    "id": 1
}

# 3. Tool call: the client calls read_temperature for M03
request_tool_call = {
    "jsonrpc": "2.0",
    "method": "tools/call",
    "params": {
        "name": "read_temperature",
        "arguments": {"machine_id": "M03"}
    },
    "id": 2
}

# 4. Server response
response_tool_call = {
    "jsonrpc": "2.0",
    "result": {
        "content": [
            {
                "type": "text",
                "text": json.dumps({"machine_id": "M03", "temperature": 74.2, "unit": "°C", "state": "HOT"})
            }
        ]
    },
    "id": 2
}

print("=== MCP message examples ===")
print("\n1. Tool discovery request (client → server):")
print(json.dumps(request_tools_list, indent=2, ensure_ascii=False))
print("\n2. Tool call (client → server):")
print(json.dumps(request_tool_call, indent=2, ensure_ascii=False))
print("\n3. Server response:")
print(json.dumps(response_tool_call, indent=2, ensure_ascii=False))

## 1.4 MCP primitives: tools, resources, prompts

MCP defines three types of primitives that a server can expose:

| Primitive | Purpose | Industrial example | Controlled by |
|---|---|---|---|
| **Tool** | Executable function with side effects | `read_temperature()`, `trigger_alarm()` | LLM (model-controlled) |
| **Resource** | Readable data source (no side effect) | OPC UA address space, sensor time series | Application (app-controlled) |
| **Prompt** | Reusable prompt template | Diagnosis prompt for the HOT state | User (user-controlled) |

In this notebook we concentrate on **tools**, since they are the most common
integration method for industrial agent systems.

**Safety note**: since tools can have side effects
(machine stop, alarm triggering), MCP recommends an explicit user confirmation
before execution — the human-in-the-loop pattern from the LangGraph notebook.


---
# Part 2 – MCP server: implementation

## 2.1 Architecture decision: HTTP-based MCP server

MCP supports two transport layers:

| Transport | Use | Advantage |
|---|---|---|
| **stdio** | Local processes (Claude Desktop, CLI) | Simple, no network |
| **HTTP/SSE** | Remote servers, microservices, IIoT | Scalable, network-capable |

For industrial systems **HTTP/SSE** (server-sent events) is the relevant variant:
OPC UA servers, MES systems and databases typically run as network services.

We implement the server with **FastAPI** and the `mcp` SDK.
This corresponds to the production pattern for an MCP server that wraps an OPC UA endpoint.

## 2.2 Server implementation


In [ ]:
# ── Machine registry: simulated OPC UA address space ─────────────────────────
# In a production system this section would be replaced by asyncua calls:
#   from asyncua import Client
#   async with Client("opc.tcp://localhost:4840") as client:
#       node = client.get_node("ns=2;i=1001")
#       temp = await node.read_value()

MACHINE_REGISTRY: Dict[str, Dict[str, Any]] = {
    f"M{i+1:02d}": {"temperature": 20.0, "state": "OK", "repair_count": 0}
    for i in range(5)
}
THRESHOLD_HOT  = 70.0
THRESHOLD_WARM = 52.5   # 0.75 × 70
MCP_SERVER_PORT = 8766


# ── FastAPI application as MCP server stub ───────────────────────────────────
# Note: the official mcp SDK offers a higher-level API (mcp.server.Server).
# Here we implement the JSON-RPC layer manually to make the protocol
# transparent. In production use you would use the SDK.

mcp_app = FastAPI(title="MCP server: factory telemetry")


# ── Tool definitions (schema) ─────────────────────────────────────────────────
TOOL_DEFINITIONS = [
    {
        "name": "list_machines",
        "description": "Returns a list of all machines with their current state.",
        "inputSchema": {"type": "object", "properties": {}}
    },
    {
        "name": "read_temperature",
        "description": (
            "Reads the current temperature of a machine in °C. "
            "Also returns the classified state (OK/WARM/HOT)."
        ),
        "inputSchema": {
            "type": "object",
            "properties": {
                "machine_id": {"type": "string", "description": "Machine identifier, e.g. 'M01'"}
            },
            "required": ["machine_id"]
        }
    },
    {
        "name": "write_temperature",
        "description": (
            "Sets the temperature of a machine (e.g. after a repair). "
            "Corresponds to an OPC UA write_value() call."
        ),
        "inputSchema": {
            "type": "object",
            "properties": {
                "machine_id": {"type": "string"},
                "temperature": {"type": "number", "description": "New temperature value in °C"}
            },
            "required": ["machine_id", "temperature"]
        }
    },
    {
        "name": "simulate_tick",
        "description": (
            "Simulates one time step: all machines receive a random "
            "temperature drift. Corresponds to one MESA simulation step."
        ),
        "inputSchema": {"type": "object", "properties": {}}
    }
]


# ── Tool execution logic ──────────────────────────────────────────────────────
def classify_state(temp: float) -> str:
    if temp > THRESHOLD_HOT:
        return "HOT"
    elif temp > THRESHOLD_WARM:
        return "WARM"
    return "OK"


def execute_tool(name: str, arguments: dict) -> dict:
    """Dispatch a tool call to the corresponding implementation."""

    if name == "list_machines":
        result = []
        for mid, data in MACHINE_REGISTRY.items():
            result.append({
                "machine_id": mid,
                "temperature": data["temperature"],
                "state": classify_state(data["temperature"]),
                "repair_count": data["repair_count"]
            })
        return {"machines": result}

    elif name == "read_temperature":
        mid = arguments["machine_id"]
        if mid not in MACHINE_REGISTRY:
            return {"error": f"Machine '{mid}' not found."}
        temp = MACHINE_REGISTRY[mid]["temperature"]
        return {
            "machine_id": mid,
            "temperature": round(temp, 1),
            "unit": "°C",
            "state": classify_state(temp)
        }

    elif name == "write_temperature":
        mid = arguments["machine_id"]
        new_value = float(arguments["temperature"])
        if mid not in MACHINE_REGISTRY:
            return {"error": f"Machine '{mid}' not found."}
        MACHINE_REGISTRY[mid]["temperature"] = new_value
        if new_value <= 20.0:
            MACHINE_REGISTRY[mid]["repair_count"] += 1
        return {"machine_id": mid, "temperature": new_value, "status": "written"}

    elif name == "simulate_tick":
        updated = []
        for mid, data in MACHINE_REGISTRY.items():
            temp = data["temperature"]
            if classify_state(temp) != "HOT":
                temp += random.uniform(-1.0, 5.0)
            MACHINE_REGISTRY[mid]["temperature"] = round(temp, 1)
            updated.append({"machine_id": mid, "temperature": round(temp, 1),
                            "state": classify_state(temp)})
        return {"tick": "executed", "machines": updated}

    else:
        return {"error": f"Unknown tool: '{name}'"}


# ── JSON-RPC 2.0 endpoint ─────────────────────────────────────────────────────
class JsonRpcRequest(BaseModel):
    jsonrpc: str = "2.0"
    method: str
    params: dict = {}
    id: Optional[int] = None


@mcp_app.post("/mcp")
def mcp_endpoint(request: JsonRpcRequest):
    """Central MCP endpoint. Handles all JSON-RPC 2.0 messages."""

    if request.method == "initialize":
        return {
            "jsonrpc": "2.0",
            "result": {
                "protocolVersion": "2024-11-05",
                "capabilities": {"tools": {}},
                "serverInfo": {"name": "factory-mcp-server", "version": "1.0.0"}
            },
            "id": request.id
        }

    elif request.method == "tools/list":
        return {
            "jsonrpc": "2.0",
            "result": {"tools": TOOL_DEFINITIONS},
            "id": request.id
        }

    elif request.method == "tools/call":
        tool_name = request.params.get("name", "")
        arguments = request.params.get("arguments", {})
        result    = execute_tool(tool_name, arguments)
        return {
            "jsonrpc": "2.0",
            "result": {
                "content": [{"type": "text", "text": json.dumps(result, ensure_ascii=False)}]
            },
            "id": request.id
        }

    else:
        return {
            "jsonrpc": "2.0",
            "error": {"code": -32601, "message": f"Method not found: {request.method}"},
            "id": request.id
        }


# ── Start the server ──────────────────────────────────────────────────────────
server_config = uvicorn.Config(
    mcp_app, host="127.0.0.1", port=MCP_SERVER_PORT, log_level="warning"
)
mcp_server = uvicorn.Server(server_config)
mcp_thread = threading.Thread(target=mcp_server.run, daemon=True)
mcp_thread.start()
time.sleep(1.5)

# Connection test
test = requests.post(
    f"http://127.0.0.1:{MCP_SERVER_PORT}/mcp",
    json={"jsonrpc": "2.0", "method": "initialize", "params": {}, "id": 0}
)
print("MCP server started. initialize response:")
print(json.dumps(test.json(), indent=2, ensure_ascii=False))

---
# Part 3 – MCP client: tool discovery and tool call

## 3.1 Client class

The MCP client encapsulates the HTTP communication with the MCP server.
In a production environment you would use the `mcp` SDK class `ClientSession`;
here we implement a simple HTTP client to keep the protocol transparent.


In [ ]:
class MCPClient:
    """Simple MCP client for HTTP transport.
    
    Encapsulates JSON-RPC 2.0 communication with an MCP server.
    In production use: mcp.ClientSession with stdio or SSE transport.
    """

    def __init__(self, server_url: str):
        self.server_url = server_url
        self._next_id = 1
        self.available_tools: List[dict] = []

    def _next_request_id(self) -> int:
        id_ = self._next_id
        self._next_id += 1
        return id_

    def _send(self, method: str, params: dict) -> dict:
        """Send a JSON-RPC 2.0 request and return the result."""
        request = {
            "jsonrpc": "2.0",
            "method": method,
            "params": params,
            "id": self._next_request_id()
        }
        response = requests.post(self.server_url, json=request, timeout=5.0)
        response.raise_for_status()
        data = response.json()
        if "error" in data:
            raise RuntimeError(f"MCP error: {data['error']}")
        return data.get("result", {})

    def initialize(self) -> dict:
        """Handshake with the MCP server."""
        return self._send("initialize", {})

    def list_tools(self) -> List[dict]:
        """Query all available tools from the server (tool discovery)."""
        result = self._send("tools/list", {})
        self.available_tools = result.get("tools", [])
        return self.available_tools

    def call_tool(self, name: str, arguments: dict = None) -> Any:
        """Call a tool on the server and return the parsed result."""
        result = self._send(
            "tools/call",
            {"name": name, "arguments": arguments or {}}
        )
        # MCP response: content[0].text contains the JSON-serialised result
        content = result.get("content", [])
        if content and content[0].get("type") == "text":
            return json.loads(content[0]["text"])
        return result

    def print_tool_schemas(self):
        """Print all tool definitions in a readable form."""
        if not self.available_tools:
            self.list_tools()
        for tool in self.available_tools:
            print(f"\nTool: {tool['name']}")
            print(f"  Description: {tool['description']}")
            schema = tool.get("inputSchema", {}).get("properties", {})
            if schema:
                print(f"  Parameters:")
                for param, info in schema.items():
                    print(f"    - {param} ({info.get('type','?')}): {info.get('description','')}")


print("MCPClient class defined.")

In [ ]:
# ── Instantiate the client and perform tool discovery ────────────────────────
client = MCPClient(f"http://127.0.0.1:{MCP_SERVER_PORT}/mcp")

print("=== MCP client: tool discovery ===")
client.initialize()
client.print_tool_schemas()

print("\n=== Manual tool calls ===")

# List all machines
machines = client.call_tool("list_machines")
print("\nlist_machines():")
for m in machines["machines"]:
    print(f"  {m['machine_id']}: {m['temperature']} °C | {m['state']}")

# Simulate one time step
tick = client.call_tool("simulate_tick")
print("\nsimulate_tick():")
for m in tick["machines"]:
    print(f"  {m['machine_id']}: {m['temperature']} °C | {m['state']}")

# Read a single machine
reading = client.call_tool("read_temperature", {"machine_id": "M01"})
print(f"\nread_temperature(M01): {reading}")

---
# Part 4 – LLM as decision-making instance

## 4.1 Architecture: LLM as CoordinatorAgent

In the previous notebooks the `coordinator_node` made decisions according to
**hard-coded rules**: HOT → schedule repair.

An LLM as coordinator, in contrast, can:
- bring in contextual knowledge (e.g. production schedule, machine importance)
- **weigh** several factors (temperature + trend + repair history)
- **justify decisions in natural language** (auditability)
- react to **unforeseen situations** that are not covered by rules

```
MCP server                    LLM coordinator
(telemetry)                   (decision logic)
     │                               │
     │  Tool: read_temperature()     │
     │◄──────────────────────────────│
     │  {"M01": 74.2, "M03": 71.1}   │
     │──────────────────────────────►│
     │                               │  Reasoning:
     │                               │  "M01 and M03 are HOT.
     │                               │   M01 has the higher temperature
     │  Tool: write_temperature()    │   → repair M01 first."
     │◄──────────────────────────────│
     │  {"M01": 20.0}                │
     │──────────────────────────────►│
```

## 4.2 Stub implementation (no API key required)

The stub simulates the behaviour of an LLM deterministically.
It returns structured JSON responses that imitate the real LLM output.
This makes it possible to test the overall architecture without API access.


In [ ]:
class LLMCoordinatorStub:
    """Deterministic LLM stub for tests without an API key.
    
    Imitates the behaviour of a real LLM:
    - Receives a prompt with machine telemetry
    - Returns a structured JSON decision
    - Adds a rationale in natural language
    
    The decision logic is deliberately somewhat more complex than
    the rule-based variant: prioritisation by temperature,
    WARM machines are considered as early-warning level.
    """

    def decide(self, machine_telemetry: List[dict]) -> dict:
        """Simulate an LLM call with machine telemetry as context.
        
        Returns:
          {
            "immediate_action": ["M01", "M03"],  # HOT machines, descending by temp
            "observation":      ["M02"],         # WARM machines
            "rationale":        "...",           # reasoning in natural language
            "risk_assessment":  "high"           # low / medium / high
          }
        """
        hot_machines  = [(m["machine_id"], m["temperature"])
                         for m in machine_telemetry if m["state"] == "HOT"]
        warm_machines = [m["machine_id"]
                         for m in machine_telemetry if m["state"] == "WARM"]

        # Typical LLM behaviour: prioritise by descending temperature
        hot_machines.sort(key=lambda x: x[1], reverse=True)
        immediate_action = [mid for mid, _ in hot_machines]

        # Generate the rationale (simulated LLM output)
        if hot_machines:
            hot_list = ", ".join(
                f"{mid} ({temp:.1f} °C)" for mid, temp in hot_machines
            )
            rationale = (
                f"Critical overheating detected: {hot_list}. "
                f"Repair in descending temperature priority recommended. "
            )
            if warm_machines:
                rationale += (
                    f"Machines {', '.join(warm_machines)} are in the warning range — "
                    f"close monitoring recommended."
                )
            risk = "high" if len(hot_machines) >= 2 else "medium"
        elif warm_machines:
            rationale = (
                f"No critical machines. {', '.join(warm_machines)} in the "
                f"warning range — observe the trend."
            )
            risk = "low"
        else:
            rationale = "All machines in normal operation. No action required."
            risk = "low"

        return {
            "immediate_action": immediate_action,
            "observation": warm_machines,
            "rationale": rationale,
            "risk_assessment": risk
        }


# ── Test the stub ─────────────────────────────────────────────────────────────
stub = LLMCoordinatorStub()

test_telemetry = [
    {"machine_id": "M01", "temperature": 78.3, "state": "HOT"},
    {"machine_id": "M02", "temperature": 55.1, "state": "WARM"},
    {"machine_id": "M03", "temperature": 74.5, "state": "HOT"},
    {"machine_id": "M04", "temperature": 32.0, "state": "OK"},
    {"machine_id": "M05", "temperature": 22.1, "state": "OK"},
]

decision = stub.decide(test_telemetry)
print("=== LLM coordinator stub: decision ===")
print(json.dumps(decision, indent=2, ensure_ascii=False))

## 4.3 Real LLM API call (extension)

The following section is **commented out** and requires an Anthropic API key
(alternatively: an OpenAI API key; then use `ChatOpenAI` instead of `ChatAnthropic`).

The interface to the rest of the implementation is **identical** to the stub:
`decide(machine_telemetry) → dict`. The rest of the graph (Part 5)
does not need to be changed.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# EXTENSION: real LLM API call (API key required)
# To activate: remove the comment characters and set API_KEY
# ════════════════════════════════════════════════════════════════════════════════

# import os
# from langchain_anthropic import ChatAnthropic
# from langchain_core.messages import HumanMessage, SystemMessage
#
# ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "sk-ant-...")
#
# class LLMCoordinatorReal:
#     """Real LLM coordinator via the Anthropic Claude API.
#
#     Interface identical to the stub → direct replacement possible.
#     """
#
#     SYSTEM_PROMPT = textwrap.dedent("""
#         You are an AI coordinator agent in a manufacturing plant.
#         Your task: analyse machine telemetry and prioritise maintenance actions.
#
#         Thresholds:
#         - HOT:  temperature > 70 °C → immediate repair required
#         - WARM: temperature > 52.5 °C → observation and early warning
#         - OK:   temperature ≤ 52.5 °C → no action required
#
#         Respond EXCLUSIVELY in the following JSON format:
#         {
#             "immediate_action": ["M01", ...],
#             "observation": ["M02", ...],
#             "rationale": "...",
#             "risk_assessment": "low" | "medium" | "high"
#         }
#         No text before or after the JSON. No markdown.
#     """)
#
#     def __init__(self, api_key: str):
#         self.llm = ChatAnthropic(
#             model="claude-sonnet-4-20250514",
#             api_key=api_key,
#             max_tokens=512,
#             temperature=0.1   # low temperature for consistent JSON output
#         )
#
#     def decide(self, machine_telemetry: List[dict]) -> dict:
#         """Call the LLM with the current machine telemetry."""
#         telemetry_text = json.dumps(machine_telemetry, indent=2, ensure_ascii=False)
#         user_prompt = f"Current machine telemetry:\n{telemetry_text}\n\nDecision:"
#
#         messages = [
#             SystemMessage(content=self.SYSTEM_PROMPT),
#             HumanMessage(content=user_prompt)
#         ]
#
#         response = self.llm.invoke(messages)
#         raw_text = response.content.strip()
#
#         # Parse JSON safely
#         try:
#             return json.loads(raw_text)
#         except json.JSONDecodeError:
#             # Fallback: use the stub if JSON parsing fails
#             print(f"[WARNING] LLM output is not valid JSON: {raw_text[:100]}")
#             return LLMCoordinatorStub().decide(machine_telemetry)
#
# # Instantiation:
# # coordinator_llm = LLMCoordinatorReal(api_key=ANTHROPIC_API_KEY)
# # decision = coordinator_llm.decide(test_telemetry)
# # print(json.dumps(decision, indent=2, ensure_ascii=False))

print("Real LLM call: code ready, commented out (no API key required).")
print("To activate: remove the comment characters and set ANTHROPIC_API_KEY.")

---
# Part 5 – Full integration: LangGraph + MCP + LLM

## 5.1 System architecture

```
┌──────────────────────────────────────────────────────────────────────┐
│  LangGraph host                                                      │
│                                                                      │
│  START → [machine_node] → [coordinator_node] → [maintenance_node]   │
│               │                   │                    │             │
│               │ MCP:              │ LLM stub /         │ MCP:        │
│               │ simulate_tick()   │ ChatAnthropic       │ write_temp()│
│               │ list_machines()   │                    │             │
│               ▼                   ▼                    ▼             │
│         MCP server            Decision             MCP server        │
│         (telemetry)           (JSON)               (actuator)        │
└──────────────────────────────────────────────────────────────────────┘
```

## 5.2 Extended GraphState


In [ ]:
class FullGraphState(TypedDict):
    """Extended state for the full integration.
    
    Additions compared with the LangGraph notebook:
    - llm_rationale: reasoning output of the LLM coordinator
    - risk_assessment: LLM assessment of the overall risk
    - watch_list: WARM machines (LLM early warning)
    - mcp_log: log of all MCP tool calls
    """
    # Machine states (from the MCP server)
    telemetry: List[dict]             # raw data from the MCP server

    # Coordinator decisions (LLM output)
    repair_queue: List[str]
    watch_list: List[str]             # NEW: WARM machines
    llm_rationale: str                # NEW: LLM reasoning
    risk_assessment: str              # NEW: low / medium / high
    coordinator_log: List[str]

    # Maintenance log
    maintenance_log: List[str]

    # MCP log
    mcp_log: List[str]                # NEW: all tool calls

    # Control
    cycle: int
    max_cycles: int
    next_action: str


def full_initial_state(max_cycles: int = 8) -> FullGraphState:
    return FullGraphState(
        telemetry=[],
        repair_queue=[], watch_list=[],
        llm_rationale="", risk_assessment="unknown",
        coordinator_log=[], maintenance_log=[], mcp_log=[],
        cycle=0, max_cycles=max_cycles, next_action="machine_node",
    )

print("FullGraphState defined.")

In [ ]:
# ── Coordinator instance (stub or real) ──────────────────────────────────────
# Using the stub here; for a real API call replace it by LLMCoordinatorReal:
# coordinator = LLMCoordinatorReal(api_key=ANTHROPIC_API_KEY)
coordinator = LLMCoordinatorStub()

# ── MCP client instance ───────────────────────────────────────────────────────
mcp = MCPClient(f"http://127.0.0.1:{MCP_SERVER_PORT}/mcp")
mcp.initialize()


# ── machine_node: fetch telemetry via MCP ────────────────────────────────────
def full_machine_node(state: FullGraphState) -> dict:
    """Fetch machine telemetry via MCP tool calls.
    
    MCP calls:
      1. simulate_tick()  → update temperatures (simulates the OPC UA time series)
      2. list_machines()  → read all states
    """
    cycle = state["cycle"]
    mcp_log = list(state["mcp_log"])

    # 1. Simulate a time step (corresponds to: updating OPC UA values in the PLC)
    mcp.call_tool("simulate_tick")
    mcp_log.append(f"[Cycle {cycle}] MCP: simulate_tick() executed")

    # 2. Read telemetry
    result = mcp.call_tool("list_machines")
    telemetry = result["machines"]
    mcp_log.append(f"[Cycle {cycle}] MCP: list_machines() → {len(telemetry)} machines")

    # Console output
    print(f"  [MachineNode] Cycle {cycle}: telemetry retrieved")
    for m in telemetry:
        marker = " ⚠" if m["state"] == "HOT" else (" ○" if m["state"] == "WARM" else "")
        print(f"    {m['machine_id']}: {m['temperature']} °C [{m['state']}]{marker}")

    return {"telemetry": telemetry, "mcp_log": mcp_log, "next_action": "coordinator_node"}


# ── coordinator_node: LLM decision ───────────────────────────────────────────
def full_coordinator_node(state: FullGraphState) -> dict:
    """LLM coordinator: analyse telemetry and make the repair decision.
    
    The LLM call (stub or real) is performed in this node.
    The result contains a prioritised repair plan + rationale.
    """
    cycle = state["cycle"]
    log = list(state["coordinator_log"])

    # Obtain the LLM decision
    decision = coordinator.decide(state["telemetry"])

    repair_queue    = decision.get("immediate_action", [])
    observation     = decision.get("observation", [])
    rationale       = decision.get("rationale", "")
    risk_assessment = decision.get("risk_assessment", "unknown")

    log.append(
        f"[Cycle {cycle}] [LLM-COORD] Risk: {risk_assessment} | "
        f"Repair: {repair_queue} | Observation: {observation}"
    )
    log.append(f"[Cycle {cycle}] [LLM-COORD] Rationale: {rationale}")

    print(f"  [CoordinatorNode] LLM decision: risk={risk_assessment}")
    print(f"    Repair queue: {repair_queue}")
    print(f"    Rationale: {rationale[:80]}..." if len(rationale) > 80 else
          f"    Rationale: {rationale}")

    next_action = "maintenance_node" if repair_queue else "advance_cycle"

    return {
        "repair_queue": repair_queue,
        "watch_list": observation,
        "llm_rationale": rationale,
        "risk_assessment": risk_assessment,
        "coordinator_log": log,
        "next_action": next_action,
    }


# ── maintenance_node: write the repair via MCP ───────────────────────────────
def full_maintenance_node(state: FullGraphState) -> dict:
    """Repair the first machine in the queue and write the reset via MCP.
    
    MCP call:
      write_temperature(machine_id, 20.0) → corresponds to OPC UA write_value()
    """
    queue   = list(state["repair_queue"])
    log     = list(state["maintenance_log"])
    mcp_log = list(state["mcp_log"])
    cycle   = state["cycle"]

    if not queue:
        return {"next_action": "advance_cycle"}

    target = queue.pop(0)

    # Write the repair via MCP tool
    result = mcp.call_tool("write_temperature", {"machine_id": target, "temperature": 20.0})
    mcp_log.append(
        f"[Cycle {cycle}] MCP: write_temperature({target}, 20.0) → {result.get('status','?')}"
    )

    entry = f"[Cycle {cycle}] [MAINT] {target} repaired (MCP write access confirmed)."
    log.append(entry)
    print(f"  [MaintenanceNode] {entry}")

    return {
        "repair_queue": queue,
        "maintenance_log": log,
        "mcp_log": mcp_log,
        "next_action": "advance_cycle",
    }


def full_advance_cycle(state: FullGraphState) -> dict:
    print(f"\n─── End of cycle {state['cycle']} ───\n")
    return {"cycle": state["cycle"] + 1, "next_action": "machine_node"}


print("All nodes for the full integration defined.")

In [ ]:
# ── Build the graph ───────────────────────────────────────────────────────────
def full_router(state: FullGraphState):
    if state["cycle"] >= state["max_cycles"]:
        return "__end__"
    return state["next_action"]


full_builder = StateGraph(FullGraphState)
full_builder.add_node("machine_node",     full_machine_node)
full_builder.add_node("coordinator_node", full_coordinator_node)
full_builder.add_node("maintenance_node", full_maintenance_node)
full_builder.add_node("advance_cycle",    full_advance_cycle)
full_builder.set_entry_point("machine_node")

targets_full = {
    "machine_node":     "machine_node",
    "coordinator_node": "coordinator_node",
    "maintenance_node": "maintenance_node",
    "advance_cycle":    "advance_cycle",
    "__end__":          END,
}
for node in ["machine_node", "coordinator_node", "maintenance_node", "advance_cycle"]:
    full_builder.add_conditional_edges(node, full_router, targets_full)

full_graph = full_builder.compile()

print("=" * 60)
print("FULL INTEGRATION: LangGraph + MCP + LLM")
print("=" * 60 + "\n")

full_final_state = full_graph.invoke(full_initial_state(max_cycles=60))

print("\n" + "=" * 60)
print("SIMULATION FINISHED")
print("=" * 60)

In [ ]:
# ── Evaluation ────────────────────────────────────────────────────────────────
print("\n=== LLM coordinator log ===")
for e in full_final_state["coordinator_log"]:
    print(" ", e)

print("\n=== Maintenance log ===")
for e in full_final_state["maintenance_log"]:
    print(" ", e)

print("\n=== MCP tool log ===")
for e in full_final_state["mcp_log"]:
    print(" ", e)

print("\n=== Final state on the MCP server ===")
final_machines = mcp.call_tool("list_machines")
print(f"{'Machine':<10} {'Temperature':>13} {'State':<8} {'Repairs'}")
print("-" * 48)
for m in final_machines["machines"]:
    print(f"{m['machine_id']:<10} {m['temperature']:>10.1f} °C "
          f"{m['state']:<8} {m['repair_count']}×")

---
# Part 6 – Comparison: rule-based vs. LLM-based

## 6.1 Systematic comparison


In [ ]:
# Output as a structured table in the notebook
comparison = [
    ("Basis of decision",
     "Hard-coded thresholds (temp > 70 °C)",
     "Contextual reasoning over all factors"),
    ("Prioritisation",
     "FIFO (order of entry into the queue)",
     "Descending temperature + contextual"),
    ("Justifiability",
     "Implicit (code = logic)",
     "Explicit (natural-language rationale)"),
    ("Unknown situations",
     "Not covered → no action",
     "Reasoning by analogy from training data"),
    ("Determinism",
     "Fully deterministic",
     "Non-deterministic (temperature > 0)"),
    ("Latency per decision",
     "< 1 ms",
     "500 ms – 5 s (API round trip)"),
    ("Cost",
     "No API cost",
     "Token-based API cost"),
    ("Adaptability",
     "Code change required",
     "Prompt engineering is sufficient"),
    ("Auditability",
     "Complete (deterministically traceable)",
     "Partial (reasoning text, but not formally verifiable)"),
    ("Suitability",
     "Real-time control, safety-critical systems",
     "Diagnosis, prioritisation, explanation"),
]

print(f"{'Dimension':<28} {'Rule-based':<44} {'LLM-based'}")
print("─" * 120)
for dim, rule, llm in comparison:
    print(f"{dim:<28} {rule:<44} {llm}")

## 6.2 Recommendation: hybrid architecture

For industrial production systems a **three-layer architecture** is recommended:

```
Layer 1 – Real-time control (rule-based, < 1 ms)
─────────────────────────────────────────────────────
  temp > 80 °C → emergency stop (PLC level, no LLM)
  temp > 70 °C → set HOT flag (OPC UA)

Layer 2 – Orchestration & diagnosis (LLM-based, seconds)
──────────────────────────────────────────────────────────
  Repair prioritisation, trend analysis, rationale
  → LangGraph + MCP + LLM (this notebook)

Layer 3 – Strategic planning (LLM-based, minutes–hours)
─────────────────────────────────────────────────────────
  Maintenance planning, capacity optimisation, reporting
```

**Key principle**: LLMs do not replace the deterministic control systems
of the field level — they complement them at the orchestration level.
Safety-critical decisions (emergency stop, interlocking) always remain
in deterministic rule sets.

## 6.3 MCP as enabler: why MCP makes this architecture possible

Without MCP every one of the three layers would have to maintain its own OPC UA clients, REST adapters
and database connections — a maintenance effort that is prohibitive in industrial
environments.

MCP decouples:
- **What** is retrieved (tool name + parameters) from
- **How** it is retrieved (OPC UA, REST, SQL, MQTT)

An OPC UA MCP server can be used equally by LangGraph nodes, n8n workflows and
future A2A agents — without code changes
on the consumer side.


---
## Summary

| Concept | Implementation in this notebook | Production variant |
|---|---|---|
| **MCP server** | FastAPI + manual JSON-RPC | `mcp.server.Server` + asyncua |
| **MCP client** | `MCPClient` class with `requests` | `mcp.ClientSession` (SDK) |
| **Tool discovery** | `tools/list` → JSON schema | Automatic via SDK |
| **LLM coordinator** | `LLMCoordinatorStub` | `ChatAnthropic` / `ChatOpenAI` |
| **Structured output** | JSON prompt + `json.loads()` | Pydantic schema + LangChain |
| **LangGraph integration** | Node calls the MCP client | Identical |
| **Transport** | HTTP/REST | HTTP/SSE (MCP standard) or stdio |

---

**Further resources**

- MCP specification: https://modelcontextprotocol.io/specification
- MCP Python SDK: https://github.com/modelcontextprotocol/python-sdk
- LangChain MCP integration: https://python.langchain.com/docs/integrations/tools/mcp
- Anthropic Claude API: https://docs.anthropic.com/en/api
- LangGraph documentation: https://langchain-ai.github.io/langgraph/
